# 🎯 [OPS04_M07] 자동 할당된 교전 순위 재조정 및 긴급 수동 전환을 위한 HITL 승인 절차 설계
### Google Colab 실습 환경 (작전·전투 수행 / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/OPS04/ops04_m07_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 작전·전투 수행 (Level 3) |
| **미션 ID** | OPS04_M07 |
| **미션 주제** | 자동 할당된 교전 순위 재조정 및 긴급 수동 전환을 위한 HITL 승인 절차 설계 |
| **소요 시간** | 약 3.0시간 |
| **핵심 산출물** | `src/hitl/ops04_approval_workflow.py, docs/ops04_hitl_guide.md` |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

## 📂 2. 표준 산출물 디렉토리 체계 및 실습 팁
- **방법 A (`%%writefile` 저장 추천):** 코드 셀 첫 줄에 `%%writefile <산출물 경로>`(예: `%%writefile src/models/ops04_baseline_model.py`)를 추가해 해당 폴더에 모듈을 저장할 수 있습니다.
- **방법 B (셀 단위 논리 대응 인정):** 별도 파일 추출 없이 노트북 셀 단위로 독립 실행·검증하고 미션 10 통합을 완료해도 정상 인정됩니다.

> **⚠️ [교육용 가상 데이터셋 및 HITL 원칙]**
> - 본 노트북의 모든 좌표·비행체 유형·수치는 보안 규정을 준수하여 **교육용 가상 데이터로 생성**됩니다.
> - 실제 무기 제원·사거리·부대 위치·군사 기밀 데이터는 일체 사용하지 않습니다.
> - 산출되는 대응 순위는 지휘관 승인을 위한 **'권고'**이며 자동 승인·자동 실행은 없습니다.

---

In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 07. 자동 할당된 교전 순위 재조정 및 긴급 수동 전환을 위한 HITL 승인 절차 설계
**소요시간**: 약 3.0시간 | **이번 산출물**: `src/hitl/ops04_approval_workflow.py, docs/ops04_hitl_guide.md`

### 📋 미션 개요
**AI가 산출한 '권고' 대응 순위를 운용자·지휘관이 검토하여 승인·보류·반려·순위 변경하고, 긴급 시 수동 통제로 전환하는 인간 개입(HITL) 워크플로우를 구축합니다.**

- **배경 및 필요성:** 방공 대응 순위는 오판 시 치명적 결과를 초래하므로 자동 실행·자동 승인을 원천 차단하고, 역할별 권한 분리와 해시 체인 기반 감사 추적을 통해 지휘 책임 분계선을 명확히 해야 합니다.
- **핵심 과업:** AI(권고)–운용자(보류·반려)–지휘관(승인·순위 변경·수동 전환) 권한 매트릭스를 구현하고, 권한 미달·근거 부족·Low 등급·중복 조치를 거부하되 거부 이력까지 해시 체인 감사 로그에 기록하며, 보류 기한 초과 시 상급 상신 로직을 검증합니다.
- **학습 목표 (습득 역량):** 군사적 HITL 책임 분계 및 RBAC 권한 통제 설계 역량, 해시 체인 기반 무결성 감사 로그(Audit Log) 구현 능력, Fail-safe 상신 절차 설계 역량을 체득합니다.
- **검증 및 산출물:** 승인/반려 로깅 성공률 100%, 자동 승인 0건 및 위·변조 검출을 통과한 워크플로우 코드(`src/hitl/ops04_approval_workflow.py`)와 운용 가이드(`docs/ops04_hitl_guide.md`)를 산출합니다.

### 🎓 학습 목표 (습득 역량)
**[습득 역량]** AI 권고와 인간 최종 결심 간의 지휘 책임 분계선을 코드 수준으로 강제하고, 권한 통제·상급 상신·해시 체인 감사 로그를 결합한 안전한 HITL 승인 워크플로우 설계 역량을 체득합니다.

**[핵심 과업]** AI(권고)–운용자(보류·반려)–지휘관(승인·순위 변경·수동 전환) 역할 분리와 거부 규칙(근거 부족·권한 없음·Low 등급·기종결)을 구현하고, 거부 시도 포함 해시 체인 감사 로그·보류 기한 초과 상신·위·변조 검출을 검증합니다.

- **역할 기반 권한 분리(RBAC) 및 HITL 통제 설계 역량:** AI는 권고만 생성하고 운용자와 지휘관의 결심 권한을 엄격히 구분해 자동 승인 0건·긴급 수동 전환 가능 체계를 구축하는 안전 설계 원칙을 익힙니다.
- **예외 거부 및 보류 기한 초과 상신(Fail-safe) 구현 능력:** 사유 근거 미달, 비권한자 조작, Low 신뢰도 승인 시도 등을 즉시 차단하고, 보류 시간이 지나도 자동 승인 없이 상급 보고로 전환하는 방어적 상태 전이 로직을 습득합니다.
- **해시 체인(Hash-chain) 무결성 감사 추적 역량:** 승인뿐 아니라 거부된 조작 시도까지 암호화 해시 체인으로 연쇄 기록해 로깅 성공률 100%와 사후 기록 위·변조 검출을 입증하는 보안 감사 능력을 기릅니다.

### 📖 미션 배경 및 시나리오
#### 미션 배경 및 당위성
자동 계산된 요격 우선순위를 방공 지휘관이 전술적 판단에 따라 긴급 수동 변경하고 승인하는 HITL 교전 통제 워크플로우를 연계한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **자동 계산된 대응 순위를 지휘관이 승인·보류·반려·순위 변경하고, 긴급 시 수동 통제로 전환하는 HITL 절차를 만든다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **AI는 권고만 하고, 운용자는 보류·반려, 지휘관은 승인·순위 변경·수동 전환 권한을 갖도록 역할을 나눈다 과정을 진행하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/hitl/ops04_approval_workflow.py 및 docs/ops04_hitl_guide.md` 형태의 결과물을 완성한다.

### 🤖 단계별 수행 및 AI 협업 가이드
#### Step 1. 미션 7 코드를 실행해 시나리오의 각 조치가 OK 또는 거부되는 이유를 확인한다.
#### 이 단계의 역할
미션 7의 HITL 승인 워크플로우를 실행해 시나리오 11건의 조치가 승인(OK) 또는 거부되는 규칙과 해시 체인 감사 로그를 확인합니다.

#### 해야 할 일

- `ROLES`, `PERM`, `HOLD_LIMIT_S` 설정을 통해 AI(권고)–운용자(보류·반려)–지휘관(승인·순위 변경·수동 전환)의 권한 매트릭스를 파악한다.
- AI를 Co-pilot으로 활용해 거부 규칙(권한 없음, 근거 부족, Low 등급 승인 시도, 기종결 중복 조치) 4건이 모두 차단·로깅되는지 점검한다.
- 시도 11건 로깅 성공률 100%, 자동 승인 0건, 거부 4건 기록 여부를 확인한다.

#### AI 협업 가이드 (프롬프트 & 검증 지침)

  - **프롬프트 질의 예시:** "운용자와 지휘관 권한(PERM), 근거 최소 길이, Low 등급 승인 금지, 기종결 중복 차단 규칙을 적용하고, 승인뿐 아니라 거부된 시도까지 SHA-256 해시 체인 감사 로그에 남기는 HITL 워크플로우 검증 코드를 검토해줘."

  - **AI 결과 검증 포인트:** AI가 거부된 시도는 감사 로그에 남기지 않고 예외(`raise`)로 빠뜨리지 않았는지, 그리고 자동 승인 분기가 완전히 제거(자동 승인 0건)되었는지 코드로 확인합니다.

#### 핵심 확인 지표

> 조치 시도 11건 로깅 성공률 100%, 자동 승인 0건, 거부 4건(권한·근거·Low 등급·중복) 기록

#### 학습자가 제출해야 하는 내용

- 미션 7 시나리오 11건 실행 결과 및 감사 로그 출력본
- AI 프롬프트 질의 및 RBAC/거부 로깅 검증 기록

#### Step 2. 검토 대기열에서 상신된 보류 항목이 맨 위로 오는지 본다.
#### 이 단계의 역할
보류 기한(`HOLD_LIMIT_S`)을 초과한 항목이 자동 승인되지 않고 검토 대기열 최상단으로 상급 상신되는지 검증합니다.

#### 해야 할 일

- 보류 처리된 항적의 경과 시간이 `HOLD_LIMIT_S`를 넘겼을 때 상태 전이와 대기열 정렬 순서를 확인한다.
- AI에게 보류 기한 초과 항목이 자동 승인 대신 '상신(ESCALATED)' 상태로 전환되어 대기열 1순위로 올라오는지 검증하는 단위 테스트를 요청한다.
- 대기열 출력에서 상신된 보류 항목의 최우선 배치와 대기시간을 확인한다.

#### AI 협업 가이드 (프롬프트 & 검증 지침)

  - **프롬프트 질의 예시:** "HITL 검토 대기열에서 보류 기한(HOLD_LIMIT_S)을 초과한 항목이 절대 자동 승인되지 않고 상급 상신 상태로 전환되어 검토 대기열 맨 위로 정렬되는지 검증하는 단위 테스트를 작성해줘."

  - **AI 결과 검증 포인트:** AI가 타임아웃 처리 관성으로 `auto_approve`를 구현하지 않았는지 철저히 점검하고, 상신 이력 역시 해시 체인 감사 로그에 누락 없이 기록되는지 확인합니다.

#### 핵심 확인 지표

> 보류 초과 자동 승인 0건, 상신 항목 검토 대기열 최상단 배치 확인

#### 학습자가 제출해야 하는 내용

- 검토 대기열 상신 정렬 확인 로그 및 단위 테스트 결과
- AI 작성 테스트의 타임아웃 안전성 검증 기록

#### Step 3. 감사 로그 위·변조 시험 결과를 확인한다.
#### 이 단계의 역할
감사 로그의 해시 체인(`SHA-256`) 무결성 검증과 고의 변조(4번 기록) 탐지 시험 결과를 확인합니다.

#### 해야 할 일

- 각 로그 레코드의 `prev_hash`와 현재 해시 계산식이 체인으로 연결되는 구조를 점검한다.
- AI에게 4번 로그 레코드의 조치 내용이나 시각을 임의로 변조했을 때 체인 검증 함수가 정확한 변조 인덱스를 적발하는지 확인하는 설명과 테스트를 요청한다.
- 위·변조 검출 성공과 `docs/ops04_hitl_guide.md` 운용 절차 정리를 완료한다.

#### AI 협업 가이드 (프롬프트 & 검증 지침)

  - **프롬프트 질의 예시:** "해시 체인 감사 로그의 4번 레코드 내용을 임의로 수정한 뒤 verify_chain()을 실행해 위·변조된 레코드 번호가 즉시 검출되는지 확인하고, 지휘관/운용자용 HITL 운용 가이드(ops04_hitl_guide.md) 핵심 절차를 정리해줘."

  - **AI 결과 검증 포인트:** 해시 계산 시 JSON 직렬화 키 순서(`sort_keys=True`)가 고정되어 재실행 시 거짓 위·변조 오탐이 발생하지 않는지, 변조 시에는 100% 검출되는지 확인합니다.

#### 핵심 확인 지표

> 해시 체인 위·변조 검출 성공(4번 기록 적발), 로깅 성공률 100%

#### 학습자가 제출해야 하는 내용

- 감사 로그 위·변조 시험 결과 및 HITL 운용 가이드(`docs/ops04_hitl_guide.md` 초안)
- AI 작성 해시 검증 로직 및 가이드 문서 대조 기록

#### Step 4. 도전 과제 및 파라미터 튜닝 최적화
#### 이 단계의 역할
시나리오에 운용자(`OPR-09`)의 수동 전환(`MODE`) 무단 시도 줄을 추가해 권한 거부 및 감사 로그 기록을 검증합니다.

#### 해야 할 일

- 테스트 시나리오에 운용자(`OPR-09`)가 지휘관 전용 권한인 `MODE`(긴급 수동 전환)를 요청하는 행을 추가한다.
- AI에게 해당 시도가 권한 부족으로 거부되면서도 거부 사유와 함께 해시 체인에 추가되는지 검증하는 스니펫을 요청한다.
- 거부 응답 및 해시 체인 무결성 유지 결과를 비교 메모로 제출한다.

#### AI 협업 가이드 (프롬프트 & 검증 지침)

  - **프롬프트 질의 예시:** "HITL 시나리오에 운용자(OPR-09)가 MODE(수동 전환)를 시도하는 이벤트를 추가해 실행하고, 해당 시도가 거부되는 동시에 해시 체인 감사 로그에 정상 기록되는지 확인하는 테스트 코드를 작성해줘."

  - **AI 결과 검증 포인트:** 권한 거부 이후에도 기존 감사 로그의 해시 체인이 끊어지지 않고 다음 레코드까지 `verify_chain()`이 통과하는지 확인합니다.

#### 핵심 확인 지표

> OPR-09 MODE 시도 즉시 거부 및 해시 체인 감사 로그 무결성 유지 확인

#### 학습자가 제출해야 하는 내용

- 권한 침범 시나리오 추가 전후 감사 로그 및 비교 분석 메모
- AI 질의 프롬프트 및 권한 거부/체인 무결성 검증 기록

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정
**평가 구분:** 필수

**핵심 평가영역:** 지휘 책임 분계선 명확화 및 성급한 자동 단정 방지 체계 부합성

#### 미션 PASS 조건
- 정량 기준: 인간 검토 큐(Queue) 대기시간 및 승인/반려 처리 로깅 성공률 100%
- 진행 기준: 승인/반려 인터페이스 동작 및 감사 로그(Audit Log) 생성 검증
- 검증 방법: 모의 조작자 승인/반려 시뮬레이션 및 데이터 무결성 검증
- AI 검증 기준: AI 생성 HITL 코드의 타임아웃 자동 승인 차단, 거부 시도 포함 해시 체인 로깅 및 변조 적발을 단위 테스트로 검증 완료
#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 AI 프롬프트 설계의 구체성, AI 산출물의 오류·보안 위반을 인간 검토(HITL)로 식별·수정한 기록, 그리고 출력 수치의 군 실무 부합성을 종합 채점한다.

In [ ]:
"""자동 계산된 대응 우선순위를 방공 지휘관이 승인·보류·반려하거나 수동으로 순위를 바꾸고, 긴급 시 수동 통제 모드로 전환하는 인간 개입(HITL) 승인 워크플로우를 구현한다. 자동 승인은 없으며, 모든 시도(거부된 시도 포함)는 역할·근거와 함께 해시 체인 감사 로그에 기록된다."""
import hashlib
import json
import numpy as np
import pandas as pd

ROLES = {"CDR-01": "commander", "OPR-07": "operator", "OPR-09": "operator"}          # 가상 운용자
PERM = {"APPROVE": {"commander"}, "REORDER": {"commander"}, "MODE": {"commander"},
        "HOLD": {"commander", "operator"}, "REJECT": {"commander", "operator"}}
HOLD_LIMIT_S = 30


class FakeClock:
    t = 0.0

    def __call__(self):
        return self.t


class HitlBoard:
    def __init__(self, clock):
        self.clock, self.items, self.log, self.mode = clock, {}, [], "RECOMMEND"
        self.attempts = 0

    def _record(self, body):
        prev = self.log[-1]["hash"] if self.log else "GENESIS"
        body = {**body, "ts": self.clock(), "mode": self.mode}
        h = hashlib.sha256((prev + json.dumps(body, ensure_ascii=False, sort_keys=True)).encode()).hexdigest()
        self.log.append({**body, "prev": prev, "hash": h})

    def verify(self):
        prev = "GENESIS"
        for n, r in enumerate(self.log, 1):
            body = {k: v for k, v in r.items() if k not in ("prev", "hash")}
            if r["prev"] != prev or r["hash"] != hashlib.sha256((prev + json.dumps(body, ensure_ascii=False, sort_keys=True)).encode()).hexdigest():
                return False, n
            prev = r["hash"]
        return True, -1

    def submit(self, tid, ai_rank, tti, grade):
        self.items[tid] = {"id": tid, "ai_rank": ai_rank, "rank": ai_rank, "tti": tti, "grade": grade, "status": "PENDING",
                           "t_in": self.clock(), "t_out": None, "hold_at": None, "escalated": False}
        self._record({"event": "SUBMIT", "id": tid, "ai_rank": ai_rank, "by": "AI"})

    def act(self, op, action, tid=None, reason="", new_rank=None):
        """모든 시도는 성공·거부와 무관하게 감사 로그에 남긴다."""
        self.attempts += 1
        err = None
        role = ROLES.get(op)
        it = self.items.get(tid)
        if role is None or role not in PERM.get(action, set()):
            err = f"권한 없음({op}:{action})"
        elif len(reason.strip()) < 6:
            err = "판단 근거 6자 이상 필요"
        elif action != "MODE" and (it is None or it["status"] in ("APPROVED", "REJECTED")):
            err = "대상 없음 또는 이미 종결"
        elif action == "APPROVE" and it["grade"] == "Low":
            err = "Low 등급은 승인 전 추가 확인(HOLD) 필요"
        if err:
            self._record({"event": "REFUSED", "action": action, "id": tid, "operator": op, "reason": reason, "error": err})
            return err
        if action == "MODE":
            self.mode = "MANUAL" if self.mode == "RECOMMEND" else "RECOMMEND"
        elif action == "REORDER":
            it["rank"] = new_rank
        else:
            it["status"] = {"APPROVE": "APPROVED", "HOLD": "HOLD", "REJECT": "REJECTED"}[action]
            it["hold_at"] = self.clock() if action == "HOLD" else None
            if action != "HOLD":
                it["t_out"] = self.clock()
        self._record({"event": action, "id": tid, "operator": op, "role": role, "reason": reason, "new_rank": new_rank})
        return "OK"

    def tick(self):
        """보류 기한 초과 시 자동 승인하지 않고 지휘관에게 상신(ESCALATE)만 한다."""
        for it in self.items.values():
            if it["status"] == "HOLD" and not it["escalated"] and self.clock() - it["hold_at"] > HOLD_LIMIT_S:
                it["escalated"] = True
                self._record({"event": "ESCALATE", "id": it["id"], "by": "SYSTEM"})

    def queue(self):
        todo = [i for i in self.items.values() if i["status"] in ("PENDING", "HOLD")]
        return sorted(todo, key=lambda i: (not i["escalated"], i["rank"]))


clk = FakeClock()
hb = HitlBoard(clk)
rng = np.random.default_rng(5)
# [교육용가상자료] 미션 5 권고 결과를 가정한 12건
for k in range(12):
    hb.submit(f"T09{k:03d}", k + 1, int(rng.integers(60, 600)), ["High", "High", "Med", "Low"][k % 4])
scenario = [  # (경과 s, 운용자, 조치, 표적, 근거, 새 순위)
    (2, "OPR-07", "APPROVE", "T09000", "도달 임박 표적", None),                         # 권한 없음 -> 거부
    (3, "CDR-01", "APPROVE", "T09000", "복수 센서 확인, 지휘소 지향", None),
    (5, "CDR-01", "REORDER", "T09005", "탄약고 지향 확인, 우선 대응 필요", 2),
    (6, "OPR-09", "HOLD", "T09003", "단일 센서, 육안 확인 요청", None),
    (7, "CDR-01", "APPROVE", "T09003", "급함", None),                                   # 근거 부족 -> 거부
    (8, "OPR-07", "REJECT", "T09007", "민간 응답 신호 확인, 오경보", None),
    (10, "CDR-01", "MODE", None, "통신 지연으로 긴급 수동 통제 전환", None),
    (12, "CDR-01", "APPROVE", "T09001", "수동 통제 중 지휘관 직접 판단", None),
    (15, "CDR-01", "APPROVE", "T09011", "Low 등급이나 도달 임박 판단", None),              # Low 승인 -> 거부
    (20, "CDR-01", "APPROVE", "T09001", "중복 승인 시도 확인용", None),                    # 이미 종결 -> 거부
    (25, "CDR-01", "MODE", None, "통신 복구, 권고 모드 복귀", None),
]
for t, op, a, tid, rsn, nr in scenario:
    clk.t = t; hb.tick()
    res = hb.act(op, a, tid, rsn, nr)
    print(f"[t={t:>3}s] {op} {a:8s} {tid or '-':7s} -> {res}")
clk.t = 45; hb.tick()

print("\n[검토 대기열] (상신 우선, 그다음 순위)")
print(pd.DataFrame(hb.queue())[["id", "ai_rank", "rank", "grade", "status", "escalated"]].head(6).to_string(index=False))
auto = sum(1 for r in hb.log if r["event"] == "APPROVE" and r.get("operator") in (None, "AI", "SYSTEM"))
logged = sum(1 for r in hb.log if r["event"] not in ("SUBMIT", "ESCALATE"))
ok, _ = hb.verify()
done = [i for i in hb.items.values() if i["t_out"] is not None]
waits = [i["t_out"] - i["t_in"] for i in done]
print(f"\n조치 시도 {hb.attempts}건 중 로그 기록 {logged}건 -> 로깅 성공률 {logged / hb.attempts:.0%} (목표 100%) -> {'달성' if logged == hb.attempts else '미달'}")
print(f"종결 {len(done)}건 대기시간 평균 {np.mean(waits):.1f}s / 최대 {max(waits):.0f}s, 미결 {len(hb.queue())}건, 자동 승인 {auto}건")
print(f"감사 로그 무결성: {'정상' if ok else '훼손'} ({len(hb.log)}건)")
hb.log[3]["reason"] = "사후 수정"
ok2, n2 = hb.verify()
print(f"위·변조 시험(4번째 기록 근거 수정) -> {'검출(' + str(n2) + '번 기록)' if not ok2 else '미검출'}")
print("\n[지휘 책임 분계선] AI: 순위 '권고'만 / 운용자: 보류·반려 / 지휘관: 승인·순위 변경·수동 전환")
print("승인은 지휘계통에 대응 인가를 요청하는 단계이며, 실제 교전은 교전수칙과 지휘계통에 따른다.")


<details>
<summary><b>💡 [참고] 미션 07 예상 실행 결과 (Output)</b></summary>

```text
[t=  2s] OPR-07 APPROVE  T09000  -> 권한 없음(OPR-07:APPROVE)
[t=  3s] CDR-01 APPROVE  T09000  -> OK
[t=  5s] CDR-01 REORDER  T09005  -> OK
[t=  6s] OPR-09 HOLD     T09003  -> OK
[t=  7s] CDR-01 APPROVE  T09003  -> 판단 근거 6자 이상 필요
[t=  8s] OPR-07 REJECT   T09007  -> OK
[t= 10s] CDR-01 MODE     -       -> OK
[t= 12s] CDR-01 APPROVE  T09001  -> OK
[t= 15s] CDR-01 APPROVE  T09011  -> Low 등급은 승인 전 추가 확인(HOLD) 필요
[t= 20s] CDR-01 APPROVE  T09001  -> 대상 없음 또는 이미 종결
[t= 25s] CDR-01 MODE     -       -> OK

[검토 대기열] (상신 우선, 그다음 순위)
    id  ai_rank  rank grade  status  escalated
T09003        4     4   Low    HOLD       True
T09005        6     2  High PENDING      False
T09002        3     3   Med PENDING      False
T09004        5     5  High PENDING      False
T09006        7     7   Med PENDING      False
T09008        9     9  High PENDING      False

조치 시도 11건 중 로그 기록 11건 -> 로깅 성공률 100% (목표 100%) -> 달성
종결 3건 대기시간 평균 7.7s / 최대 12s, 미결 9건, 자동 승인 0건
감사 로그 무결성: 정상 (24건)
위·변조 시험(4번째 기록 근거 수정) -> 검출(4번 기록)

[지휘 책임 분계선] AI: 순위 '권고'만 / 운용자: 보류·반려 / 지휘관: 승인·순위 변경·수동 전환
승인은 지휘계통에 대응 인가를 요청하는 단계이며, 실제 교전은 교전수칙과 지휘계통에 따른다.
```
</details>
